## Import libraries

In [2]:
import pandas as pd
import re

## Load datset

**Load gene expression**

ref: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE81089

In [3]:
# Specify the CSV file path
csv_file = "GSE81089_FPKM_cufflinks.csv"

# Read the CSV file into a DataFrame
df = pd.read_csv(csv_file)
print(f"dataset shape: {df.shape}")
print(df.head())

dataset shape: (63130, 219)
   Ensembl_gene_id      L400T      L401T      L404T     L406T     L413T  \
0  ENSG00000000003  52.195000  37.889100  23.191000  25.03240  41.96860   
1  ENSG00000000005   0.230061   0.086034   0.048022   0.00000   2.57090   
2  ENSG00000000419  43.861600  47.045700  38.129200  54.30300  51.29690   
3  ENSG00000000457  14.710100   7.812330  12.311700   8.41631   8.84999   
4  ENSG00000000460   4.813350   5.920730   8.213850   6.71221   4.79088   

       L414T      L417T     L420T     L439T  ...      L877T     L879T  \
0  28.579400  19.900800  23.26320  37.02400  ...  29.719400  28.03980   
1   0.087192   0.234047   0.00000   0.00000  ...   0.404797   0.00000   
2  42.160400  78.196100  60.72830  23.52960  ...  35.623900  39.29990   
3   6.280520   4.662350   7.39264  14.55440  ...  10.079700   6.79271   
4   9.647640   6.605090   8.49746   5.45051  ...   6.944480   5.92882   

      L880T      L881N     L881T      L884T     L885T     L886T     L887T  \
0  21

**Merge gene name**

In [4]:
# Load the GTF file (change path to your actual file)
gtf_file = "Homo_sapiens.GRCh38.77.gtf"

# Read GTF and extract gene_id and gene_name
gtf_data = pd.read_csv(gtf_file, comment='#', sep='\t', header=None)

# Extract relevant columns
attributes = gtf_data.iloc[:, 8]

# Parse gene_id and gene_name from the attributes column
gene_map = {}
for attr in attributes:
    gene_id_match = re.search(r'gene_id "([^"]+)"', attr)
    gene_name_match = re.search(r'gene_name "([^"]+)"', attr)
    
    if gene_id_match and gene_name_match:
        gene_id = gene_id_match.group(1)
        gene_name = gene_name_match.group(1)
        gene_map[gene_id] = gene_name

# Convert to DataFrame
gene_df = pd.DataFrame(list(gene_map.items()), columns=["gene_id", "gene_name"])
print(f"# of gene names: {len(gene_df)} \n {gene_df.head()}")

/var/folders/5t/qhv6pkk115b3_mpr1bbc5c9c0000gn/T/ipykernel_32863/2513924972.py:5: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  gtf_data = pd.read_csv(gtf_file, comment='#', sep='\t', header=None)


# of gene names: 64253 
            gene_id   gene_name
0  ENSG00000223972     DDX11L1
1  ENSG00000227232      WASH7P
2  ENSG00000278267   MIR6859-2
3  ENSG00000243485  MIR1302-10
4  ENSG00000274890  MIR1302-11


In [5]:
# Merge the main DataFrame with gene mapping DataFrame
merged_df = df.merge(gene_df, left_on='Ensembl_gene_id', right_on='gene_id', how='left')

# Drop unmapped genes and Ensembl_gene_id columns
filtered_df = merged_df.dropna(subset=['gene_name']).drop(columns=['gene_id', 'Ensembl_gene_id'])

print(f"# of dataset: {len(filtered_df)} \n {filtered_df.head()}")

# of dataset: 56685 
        L400T      L401T      L404T     L406T     L413T      L414T      L417T  \
0  52.195000  37.889100  23.191000  25.03240  41.96860  28.579400  19.900800   
1   0.230061   0.086034   0.048022   0.00000   2.57090   0.087192   0.234047   
2  43.861600  47.045700  38.129200  54.30300  51.29690  42.160400  78.196100   
3  14.710100   7.812330  12.311700   8.41631   8.84999   6.280520   4.662350   
4   4.813350   5.920730   8.213850   6.71221   4.79088   9.647640   6.605090   

      L420T     L439T      L440T  ...     L879T     L880T      L881N  \
0  23.26320  37.02400  101.31400  ...  28.03980  21.30900   6.603010   
1   0.00000   0.00000    0.00000  ...   0.00000   0.00000   0.149436   
2  60.72830  23.52960   34.25800  ...  39.29990  43.93240  32.476700   
3   7.39264  14.55440   12.83890  ...   6.79271   9.15611   4.754630   
4   8.49746   5.45051    3.40198  ...   5.92882   6.26774   2.326310   

      L881T      L884T     L885T     L886T     L887T     L890T  

In [6]:
# Transpose the DataFrame: sample IDs as rows, gene names as columns
transposed_df = filtered_df.set_index('gene_name').T.reset_index()
# Rename the columns to 'sample_id'
transposed_df.rename(columns={'index': 'sample_id'}, inplace=True)
print(f"dataset shape: {transposed_df.shape}")
transposed_df.head()

dataset shape: (218, 56686)


gene_name,sample_id,TSPAN6,TNMD,DPM1,SCYL3,C1orf112,FGR,CFH,FUCA2,GCLC,...,RP11-415F23.4,CYP2D6,SNORA28,Metazoa_SRP,GS1-166A23.1,XXbac-BPG252P9.9,XXbac-BPGBPG55C20.1,RP11-255P5.2,MIR4787,CTC-527H23.4
0,L400T,52.1950,0.230061,43.8616,14.71010,4.81335,7.40831,112.4260,43.9196,12.12890,...,0.215946,0.0,0.0,0.0,0.0,1.38909,0.312571,0.086774,0.0,0.000000
1,L401T,37.8891,0.086034,47.0457,7.81233,5.92073,9.83188,39.7146,60.4056,9.20525,...,0.583569,0.0,0.0,0.0,0.0,1.15011,0.050841,0.000000,0.0,0.000000
2,L404T,23.1910,0.048022,38.1292,12.31170,8.21385,9.68575,25.9596,49.0519,23.92220,...,0.088140,0.0,0.0,0.0,0.0,1.11998,0.551958,0.036071,0.0,0.000000
3,L406T,25.0324,0.000000,54.3030,8.41631,6.71221,10.92630,80.2073,40.4700,46.93690,...,1.150510,0.0,0.0,0.0,0.0,4.34570,0.319958,0.000000,0.0,0.086684
4,L413T,41.9686,2.570900,51.2969,8.84999,4.79088,8.36149,38.4429,58.1048,15.60820,...,0.946770,0.0,0.0,0.0,0.0,2.61839,0.415423,0.041381,0.0,0.095240


**Merge matrix data and gene expression by samples' id**

matrix data: stage, history diagnosis

In [7]:
# Path to the uploaded file
file_path = "GSE81089_series_matrix.txt"

# Read the file content
with open(file_path, "r") as file:
    lines = file.readlines()

# Extract stage tnm and histology data
stage_data = []
histology_data = []
samples = []

for line in lines:
    if line.startswith("!Sample_title"):
        samples = re.findall(r'"(.*?)"', line)
    if "stage tnm" in line:
        stage_data = re.findall(r'"stage tnm: (\d+)"', line)
    elif "histology" in line:
        histology_data = re.findall(r'"histology: (\d+)"', line)


# Ensure both lists have the same length
min_length = min(len(stage_data), len(histology_data))
stage_data = stage_data[:min_length]
histology_data = histology_data[:min_length]
samples = samples[:min_length]

# Create a DataFrame
df_matrix = pd.DataFrame({"sample_id": samples, "stage": stage_data, "diagnosis": histology_data})
print(f"# of rows: {min_length}")
print(df_matrix.head())

# of rows: 199
  sample_id stage diagnosis
0     L400T     3         2
1     L401T     5         2
2     L404T     3         2
3     L406T     1         1
4     L413T     5         2


In [8]:
# Merge the main DataFrame with gene matrix DataFrame
final_df = df_matrix.merge(transposed_df, left_on='sample_id', right_on='sample_id', how='left')
print(f"dataset shape: {final_df.shape}")
final_df.head()

dataset shape: (199, 56688)


,sample_id,stage,diagnosis,TSPAN6,TNMD,DPM1,SCYL3,C1orf112,FGR,CFH,...,RP11-415F23.4,CYP2D6,SNORA28,Metazoa_SRP,GS1-166A23.1,XXbac-BPG252P9.9,XXbac-BPGBPG55C20.1,RP11-255P5.2,MIR4787,CTC-527H23.4
0,L400T,3,2,52.1950,0.230061,43.8616,14.71010,4.81335,7.40831,112.4260,...,0.215946,0.0,0.0,0.0,0.0,1.38909,0.312571,0.086774,0.0,0.000000
1,L401T,5,2,37.8891,0.086034,47.0457,7.81233,5.92073,9.83188,39.7146,...,0.583569,0.0,0.0,0.0,0.0,1.15011,0.050841,0.000000,0.0,0.000000
2,L404T,3,2,23.1910,0.048022,38.1292,12.31170,8.21385,9.68575,25.9596,...,0.088140,0.0,0.0,0.0,0.0,1.11998,0.551958,0.036071,0.0,0.000000
3,L406T,1,1,25.0324,0.000000,54.3030,8.41631,6.71221,10.92630,80.2073,...,1.150510,0.0,0.0,0.0,0.0,4.34570,0.319958,0.000000,0.0,0.086684
4,L413T,5,2,41.9686,2.570900,51.2969,8.84999,4.79088,8.36149,38.4429,...,0.946770,0.0,0.0,0.0,0.0,2.61839,0.415423,0.041381,0.0,0.095240


**Statistical data**

In [9]:
# Get summary statistics
print(final_df.describe())

           TSPAN6        TNMD        DPM1       SCYL3    C1orf112         FGR  \
count  199.000000  199.000000  199.000000  199.000000  199.000000  199.000000   
mean    29.498083    1.008932   47.636485    8.056184    7.113864   11.957017   
std     16.175355    9.294233   17.352567    2.840776    3.226553    6.714507   
min      6.811220    0.000000   20.835000    3.263820    2.287580    1.384610   
25%     18.593650    0.000000   36.975200    6.298975    4.825255    7.268020   
50%     27.088300    0.000000   42.643300    7.509030    6.413630   10.755100   
75%     34.116650    0.044597   55.067000    9.141550    8.611070   15.891100   
max    120.324000  114.928000  142.630000   24.528900   22.042200   57.301500   

              CFH       FUCA2        GCLC        NFYA  ...  RP11-415F23.4  \
count  199.000000  199.000000  199.000000  199.000000  ...     199.000000   
mean    62.071540   51.825785   55.933176   18.836817  ...       0.528771   
std     35.885858   20.001301   76.4340

In [10]:
# Get the number of data per class
count_classes = final_df['diagnosis'].value_counts()
print(count_classes)

diagnosis
2    108
1     67
3     24
Name: count, dtype: int64


**Missing value**

In [12]:
miss_value = final_df.isnull().sum().any()
print(f"Missing values in the dataset: {miss_value}")

Missing values in the dataset: False


**Extensional Preprocessing**
1. Rename the column from "-" to "__" for compatable format


In [13]:
# Replace "-" with "__"
final_df = final_df.rename(columns = lambda x:re.sub('[^A-Za-z0-9_.]+', '__', x))
print(final_df.columns)

Index(['sample_id', 'stage', 'diagnosis', 'TSPAN6', 'TNMD', 'DPM1', 'SCYL3',
       'C1orf112', 'FGR', 'CFH',
       ...
       'RP11__415F23.4', 'CYP2D6', 'SNORA28', 'Metazoa_SRP', 'GS1__166A23.1',
       'XXbac__BPG252P9.9', 'XXbac__BPGBPG55C20.1', 'RP11__255P5.2', 'MIR4787',
       'CTC__527H23.4'],
      dtype='object', length=56688)


**Save preprocessed dataset**

In [14]:
# Save the final DataFrame to a CSV file
final_df.to_csv("preprocessed_dataset.csv", index=False, sep=",",header=True)